In [20]:
tcga_panels = {
    "onesplit_k2": ["PYCR1", "ETV4"],
    "onesplit_k3": ["PYCR1", "ETV4", "CD5L"],
    "onesplit_k5": ["PYCR1", "ETV4", "CD5L", "ADM2", "LGR4"],

    "stable_k2": ["PYCR1", "CD5L"],
    "stable_k3": ["PYCR1", "CD5L", "ANGPT4"],
    "stable_k5": ["PYCR1", "ANGPT4", "CD5L", "ETV4", "ADM2"],
}

In [21]:
import json

with open("../data/processed/tcga_discovered_panels.json", "w") as f:
    json.dump(tcga_panels, f, indent=2)

In [22]:
import GEOparse

gse = GEOparse.get_GEO(
    geo="GSE32863",
    destdir="../data/raw/geo",
)

30-Jun-2026 05:48:47 DEBUG utils - Directory ../data/raw/geo already exists. Skipping.
30-Jun-2026 05:48:47 INFO GEOparse - File already exist: using local version.
30-Jun-2026 05:48:47 INFO GEOparse - Parsing ../data/raw/geo/GSE32863_family.soft.gz: 
30-Jun-2026 05:48:47 DEBUG GEOparse - DATABASE: GeoMiame
30-Jun-2026 05:48:47 DEBUG GEOparse - SERIES: GSE32863
30-Jun-2026 05:48:47 DEBUG GEOparse - PLATFORM: GPL6884
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813411
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813412
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813413
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813414
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813415
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813416
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813417
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813418
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813419
30-Jun-2026 05:48:48 DEBUG GEOparse - SAMPLE: GSM813420
30-Jun-2026 05:48:48

In [23]:
gse

<SERIES: GSE32863 - 116 SAMPLES, 1 d(s)>

In [24]:
len(gse.gsms), list(gse.gsms.keys())[:5]

(116, ['GSM813411', 'GSM813412', 'GSM813413', 'GSM813414', 'GSM813415'])

In [25]:
first_gsm = list(gse.gsms.values())[0]

first_gsm.metadata.keys(), first_gsm.metadata

(dict_keys(['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_fax', 'contact_laboratory', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_state', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'series_id', 'data_row_count']),
 {'title': ['Adjacent non-tumor lung tissue 05L12_N (expression)'],
  'geo_accession': ['GSM813411'],
  'status': ['Public on Mar 21 2012'],
  'submission_date': ['Oct 11 2011'],
  'last_update_date': ['Mar 21 2012'],
  'type': ['RNA'],
  'channel_count': ['1'],
  'source_name_ch1': ['Adjacent non-tumor lung'],
  'organism_ch1': ['Homo sapiens'],
  'taxid_ch1': ['9606'],
  'characteristics

In [26]:
first_gsm = list(gse.gsms.values())[0]

first_gsm.table.head()

,ID_REF,VALUE,Detection Pval
0,ILMN_1762337,6.855344,0.527009
1,ILMN_2055271,6.843877,0.567852
2,ILMN_1736007,6.865865,0.479578
3,ILMN_2383229,6.799232,0.727273
4,ILMN_1806310,6.950455,0.185771


In [27]:
first_gsm.table.columns

Index(['ID_REF', 'VALUE', 'Detection Pval'], dtype='str')

In [28]:
import pandas as pd
import numpy as np

def extract_geo_label(gsm):
    source = gsm.metadata.get("source_name_ch1", [""])[0].lower()
    title = gsm.metadata.get("title", [""])[0].lower()
    characteristics = " ".join(gsm.metadata.get("characteristics_ch1", [])).lower()

    text = " ".join([source, title, characteristics])

    if "adjacent non-tumor" in text or "normal lung" in text or "non-tumor" in text:
        return "normal"
    elif "tumor" in text or "adenocarcinoma" in text:
        return "tumor"
    else:
        return "unknown"

geo_labels = []

for gsm_id, gsm in gse.gsms.items():
    geo_labels.append({
        "gsm_id": gsm_id,
        "title": gsm.metadata.get("title", [""])[0],
        "source_name": gsm.metadata.get("source_name_ch1", [""])[0],
        "sample_type": extract_geo_label(gsm),
    })

geo_labels = pd.DataFrame(geo_labels)

geo_labels["sample_type"].value_counts(), geo_labels.head()

(sample_type
 normal    58
 tumor     58
 Name: count, dtype: int64,
       gsm_id                                              title  \
 0  GSM813411  Adjacent non-tumor lung tissue 05L12_N (expres...   
 1  GSM813412           Lung adenocarcinoma 05L12_T (expression)   
 2  GSM813413  Adjacent non-tumor lung tissue 05L13_N (expres...   
 3  GSM813414           Lung adenocarcinoma 05L13_T (expression)   
 4  GSM813415  Adjacent non-tumor lung tissue 05L19_N (expres...   
 
                source_name sample_type  
 0  Adjacent non-tumor lung      normal  
 1      Lung adenocarcinoma       tumor  
 2  Adjacent non-tumor lung      normal  
 3      Lung adenocarcinoma       tumor  
 4  Adjacent non-tumor lung      normal  )

In [29]:
expr_cols = []

for gsm_id, gsm in gse.gsms.items():
    table = gsm.table.copy()

    # usually columns are ID_REF and VALUE
    s = table.set_index("ID_REF")["VALUE"]
    s.name = gsm_id

    expr_cols.append(s)

geo_probe_expr = pd.concat(expr_cols, axis=1).T

geo_probe_expr.shape

(116, 48803)

In [30]:
if "gsm_id" in geo_labels.columns:
    geo_labels = geo_labels.set_index("gsm_id")
elif "index" in geo_labels.columns:
    geo_labels = geo_labels.set_index("index").rename_axis("gsm_id")

geo_labels = geo_labels.loc[geo_probe_expr.index]

(geo_probe_expr.index == geo_labels.index).all(), geo_labels["sample_type"].value_counts()

(np.True_,
 sample_type
 normal    58
 tumor     58
 Name: count, dtype: int64)

In [31]:
gse.gpls.keys()

dict_keys(['GPL6884'])

In [32]:
gpl = list(gse.gpls.values())[0]

gpl.table.head()

,ID,Species,Source,Search_Key,Transcript,ILMN_Gene,Source_Reference_ID,RefSeq_ID,Unigene_ID,Entrez_Gene_ID,...,Chromosome,Probe_Chr_Orientation,Probe_Coordinates,Cytoband,Definition,Ontology_Component,Ontology_Process,Ontology_Function,Synonyms,GB_ACC
0,ILMN_1825594,Homo sapiens,Unigene,ILMN_89282,ILMN_89282,HS.388528,Hs.388528,NaN,Hs.388528,NaN,...,NaN,NaN,NaN,NaN,UI-CF-EC0-abi-c-12-0-UI.s1 UI-CF-EC0 Homo sapi...,NaN,NaN,NaN,NaN,BU678343
1,ILMN_1810803,Homo sapiens,RefSeq,ILMN_35826,ILMN_35826,LOC441782,XM_497527.2,XM_497527.2,NaN,441782.0,...,NaN,NaN,NaN,NaN,PREDICTED: Homo sapiens similar to spectrin do...,NaN,NaN,NaN,NaN,XM_497527.2
2,ILMN_1722532,Homo sapiens,RefSeq,ILMN_25544,ILMN_25544,JMJD1A,NM_018433.3,NM_018433.3,NaN,55818.0,...,2,+,86572991-86573040,2p11.2e,Homo sapiens jumonji domain containing 1A (JMJ...,nucleus [goid 5634] [evidence IEA],chromatin modification [goid 16568] [evidence ...,oxidoreductase activity [goid 16491] [evidence...,JHMD2A; JMJD1; TSGA; KIAA0742; DKFZp686A24246;...,NM_018433.3
3,ILMN_1884413,Homo sapiens,Unigene,ILMN_132331,ILMN_132331,HS.580150,Hs.580150,NaN,Hs.580150,NaN,...,NaN,NaN,NaN,NaN,hi56g05.x1 Soares_NFL_T_GBC_S1 Homo sapiens cD...,NaN,NaN,NaN,NaN,AW629334
4,ILMN_1906034,Homo sapiens,Unigene,ILMN_105017,ILMN_105017,HS.540210,Hs.540210,NaN,Hs.540210,NaN,...,NaN,NaN,NaN,NaN,wk77d04.x1 NCI_CGAP_Pan1 Homo sapiens cDNA clo...,NaN,NaN,NaN,NaN,AI818233


In [33]:
gpl.table.columns.tolist()

['ID',
 'Species',
 'Source',
 'Search_Key',
 'Transcript',
 'ILMN_Gene',
 'Source_Reference_ID',
 'RefSeq_ID',
 'Unigene_ID',
 'Entrez_Gene_ID',
 'GI',
 'Accession',
 'Symbol',
 'Protein_Product',
 'Array_Address_Id',
 'Probe_Type',
 'Probe_Start',
 'SEQUENCE',
 'Chromosome',
 'Probe_Chr_Orientation',
 'Probe_Coordinates',
 'Cytoband',
 'Definition',
 'Ontology_Component',
 'Ontology_Process',
 'Ontology_Function',
 'Synonyms',
 'GB_ACC']

Now we can collapse GEO probes into gene-level expression

In [34]:
probe_annot = gpl.table[["ID", "Symbol"]].copy()

probe_annot = probe_annot.rename(columns={
    "ID": "probe_id",
    "Symbol": "gene",
})

probe_annot.head()

,probe_id,gene
0,ILMN_1825594,NaN
1,ILMN_1810803,LOC441782
2,ILMN_1722532,JMJD1A
3,ILMN_1884413,NaN
4,ILMN_1906034,NaN


In [35]:
# Normalize gene symbols: cast to str (NaN → "nan") and trim whitespace
probe_annot["gene"] = probe_annot["gene"].astype(str).str.strip()

# Keep only probes with a usable gene symbol
probe_annot = probe_annot[
    probe_annot["gene"].notna()
    & (probe_annot["gene"] != "")      # empty after strip
    & (probe_annot["gene"] != "nan")   # missing values become this string after astype(str)
]

probe_annot.head(), probe_annot.shape

(       probe_id       gene
 1  ILMN_1810803  LOC441782
 2  ILMN_1722532     JMJD1A
 6  ILMN_1708805      NCOA3
 8  ILMN_1672526  LOC389834
 9  ILMN_2185604   C17orf77,
 (35966, 2))

In [36]:
common_probes = geo_probe_expr.columns.intersection(probe_annot["probe_id"])

len(common_probes), geo_probe_expr.shape[1]

(35966, 48803)

In [37]:
geo_probe_expr_common = geo_probe_expr[common_probes].copy()

probe_annot_common = (
    probe_annot
    .set_index("probe_id")
    .loc[common_probes]
    .reset_index()
)

probe_annot_common.head()

,ID_REF,gene
0,ILMN_1762337,7A5
1,ILMN_2055271,A1BG
2,ILMN_1736007,A1BG
3,ILMN_2383229,ACF
4,ILMN_1806310,ACF


In [38]:
geo_gene_expr_raw = geo_probe_expr_common.copy()
geo_gene_expr_raw.columns = probe_annot_common["gene"].values

geo_gene_expr_raw.shape

(116, 35966)

In [39]:
geo_gene_expr = geo_gene_expr_raw.T.groupby(level=0).mean().T

geo_gene_expr.shape

(116, 25440)

In [41]:
# gsm_id is already the index from the probe-level alignment cell above
if "gsm_id" in geo_labels.columns:
    geo_labels = geo_labels.set_index("gsm_id")

geo_labels = geo_labels.loc[geo_gene_expr.index]

geo_y = geo_labels["sample_type"].map({
    "normal": 0,
    "tumor": 1,
})

(geo_gene_expr.index == geo_labels.index).all(), geo_y.value_counts()

(np.True_,
 sample_type
 0    58
 1    58
 Name: count, dtype: int64)

In [42]:
geo_gene_expr.to_parquet("../data/processed/gse32863_X.parquet")
geo_labels.to_csv("../data/processed/gse32863_labels.csv", index=False)

In [43]:
import json

with open("../data/processed/tcga_discovered_panels.json", "r") as f:
    tcga_panels = json.load(f)

for panel_name, genes in tcga_panels.items():
    present = [g for g in genes if g in geo_gene_expr.columns]
    missing = [g for g in genes if g not in geo_gene_expr.columns]

    print(panel_name)
    print("present:", present)
    print("missing:", missing)
    print()

onesplit_k2
present: ['PYCR1', 'ETV4']
missing: []

onesplit_k3
present: ['PYCR1', 'ETV4', 'CD5L']
missing: []

onesplit_k5
present: ['PYCR1', 'ETV4', 'CD5L', 'ADM2', 'LGR4']
missing: []

stable_k2
present: ['PYCR1', 'CD5L']
missing: []

stable_k3
present: ['PYCR1', 'CD5L', 'ANGPT4']
missing: []

stable_k5
present: ['PYCR1', 'ANGPT4', 'CD5L', 'ETV4', 'ADM2']
missing: []



### External cohort validation using TCGA-discovered panels

In [44]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, balanced_accuracy_score, roc_auc_score

In [45]:
def evaluate_panel_on_geo_cv(panel_name, genes, X_geo, y_geo, n_splits=5):
    X_panel = X_geo[genes].copy()

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42,
    )

    scores = cross_validate(
        pipe,
        X_panel,
        y_geo,
        cv=cv,
        scoring={
            "balanced_accuracy": "balanced_accuracy",
            "roc_auc": "roc_auc",
        },
        return_train_score=False,
    )

    return {
        "panel_name": panel_name,
        "panel_size": len(genes),
        "genes": ",".join(genes),
        "geo_mean_bal_acc": scores["test_balanced_accuracy"].mean(),
        "geo_std_bal_acc": scores["test_balanced_accuracy"].std(),
        "geo_mean_auc": scores["test_roc_auc"].mean(),
        "geo_std_auc": scores["test_roc_auc"].std(),
    }

In [46]:
geo_cv_results = []

for panel_name, genes in tcga_panels.items():
    result = evaluate_panel_on_geo_cv(
        panel_name=panel_name,
        genes=genes,
        X_geo=geo_gene_expr,
        y_geo=geo_y,
        n_splits=5,
    )
    geo_cv_results.append(result)

geo_cv_results_df = pd.DataFrame(geo_cv_results)

geo_cv_results_df

,panel_name,panel_size,genes,geo_mean_bal_acc,geo_std_bal_acc,geo_mean_auc,geo_std_auc
0,onesplit_k2,2,"PYCR1,ETV4",0.955303,0.028788,1.000000,0.00000
1,onesplit_k3,3,"PYCR1,ETV4,CD5L",0.955303,0.028788,1.000000,0.00000
2,onesplit_k5,5,"PYCR1,ETV4,CD5L,ADM2,LGR4",0.964394,0.033846,1.000000,0.00000
3,stable_k2,2,"PYCR1,CD5L",0.973485,0.021694,1.000000,0.00000
4,stable_k3,3,"PYCR1,CD5L,ANGPT4",0.973485,0.021694,1.000000,0.00000
5,stable_k5,5,"PYCR1,ANGPT4,CD5L,ETV4,ADM2",0.973485,0.021694,0.998485,0.00303


In [47]:
tcga_internal = pd.DataFrame([
    {
        "panel_name": "onesplit_k2",
        "tcga_bal_acc": 0.975728,
        "tcga_auc": 0.993528,
    },
    {
        "panel_name": "onesplit_k3",
        "tcga_bal_acc": 0.990291,
        "tcga_auc": 1.000000,
    },
    {
        "panel_name": "onesplit_k5",
        "tcga_bal_acc": 0.995146,
        "tcga_auc": 0.999191,
    },
])

In [48]:
stable_internal = pd.DataFrame([
    {
        "panel_name": "stable_k2",
        "tcga_bal_acc": 0.977211,
        "tcga_auc": 0.996980,
    },
    {
        "panel_name": "stable_k3",
        "tcga_bal_acc": 0.981769,
        "tcga_auc": 0.998355,
    },
    {
        "panel_name": "stable_k5",
        "tcga_bal_acc": 0.978964,
        "tcga_auc": 0.999056,
    },
])

In [49]:
tcga_panel_summary = pd.concat(
    [tcga_internal, stable_internal],
    ignore_index=True,
)

external_comparison = geo_cv_results_df.merge(
    tcga_panel_summary,
    on="panel_name",
    how="left",
)

external_comparison["bal_acc_drop"] = (
    external_comparison["tcga_bal_acc"] - external_comparison["geo_mean_bal_acc"]
)

external_comparison["auc_drop"] = (
    external_comparison["tcga_auc"] - external_comparison["geo_mean_auc"]
)

external_comparison

,panel_name,panel_size,genes,geo_mean_bal_acc,geo_std_bal_acc,geo_mean_auc,geo_std_auc,tcga_bal_acc,tcga_auc,bal_acc_drop,auc_drop
0,onesplit_k2,2,"PYCR1,ETV4",0.955303,0.028788,1.000000,0.00000,0.975728,0.993528,0.020425,-0.006472
1,onesplit_k3,3,"PYCR1,ETV4,CD5L",0.955303,0.028788,1.000000,0.00000,0.990291,1.000000,0.034988,0.000000
2,onesplit_k5,5,"PYCR1,ETV4,CD5L,ADM2,LGR4",0.964394,0.033846,1.000000,0.00000,0.995146,0.999191,0.030752,-0.000809
3,stable_k2,2,"PYCR1,CD5L",0.973485,0.021694,1.000000,0.00000,0.977211,0.996980,0.003726,-0.003020
4,stable_k3,3,"PYCR1,CD5L,ANGPT4",0.973485,0.021694,1.000000,0.00000,0.981769,0.998355,0.008284,-0.001645
5,stable_k5,5,"PYCR1,ANGPT4,CD5L,ETV4,ADM2",0.973485,0.021694,0.998485,0.00303,0.978964,0.999056,0.005479,0.000571


In [50]:
geo_cv_results_df.to_csv(
    "../data/processed/gse32863_panel_cv_results.csv",
    index=False,
)

external_comparison.to_csv(
    "../data/processed/tcga_vs_gse32863_panel_comparison.csv",
    index=False,
)

### GSE32863 random-panel baseline

In [51]:
from sklearn.utils import check_random_state

def evaluate_random_geo_panels(panel_size, n_random=200, random_state=42):
    rng = check_random_state(random_state)
    all_genes = np.array(geo_gene_expr.columns)

    rows = []

    for i in range(n_random):
        random_genes = rng.choice(all_genes, size=panel_size, replace=False).tolist()

        result = evaluate_panel_on_geo_cv(
            panel_name=f"random_{panel_size}_{i}",
            genes=random_genes,
            X_geo=geo_gene_expr,
            y_geo=geo_y,
            n_splits=5,
        )

        rows.append({
            "panel_size": panel_size,
            "random_id": i,
            "geo_mean_bal_acc": result["geo_mean_bal_acc"],
            "geo_mean_auc": result["geo_mean_auc"],
            "genes": ",".join(random_genes),
        })

    return pd.DataFrame(rows)

In [52]:
geo_random_results = []

for k in [2, 3, 5]:
    print(f"Running GEO random panels for k={k}")
    df_k = evaluate_random_geo_panels(k, n_random=200, random_state=500 + k)
    geo_random_results.append(df_k)

geo_random_results_df = pd.concat(geo_random_results, ignore_index=True)
geo_random_results_df.head()

Running GEO random panels for k=2
Running GEO random panels for k=3
Running GEO random panels for k=5


,panel_size,random_id,geo_mean_bal_acc,geo_mean_auc,genes
0,2,0,0.750000,0.846591,"LOC255330,C9orf156"
1,2,1,0.465909,0.450884,"RAGE,LOC149448"
2,2,2,0.886364,0.945455,"C1orf33,AQR"
3,2,3,0.673485,0.716667,"LOC653476,S100A6"
4,2,4,0.472727,0.482576,"LOC388160,OFCC1"


In [53]:
geo_random_summary = (
    geo_random_results_df
    .groupby("panel_size")
    .agg(
        random_mean_bal_acc=("geo_mean_bal_acc", "mean"),
        random_std_bal_acc=("geo_mean_bal_acc", "std"),
        random_p95_bal_acc=("geo_mean_bal_acc", lambda x: np.percentile(x, 95)),
        random_max_bal_acc=("geo_mean_bal_acc", "max"),
        random_mean_auc=("geo_mean_auc", "mean"),
        random_p95_auc=("geo_mean_auc", lambda x: np.percentile(x, 95)),
        random_max_auc=("geo_mean_auc", "max"),
    )
    .reset_index()
)

geo_random_summary

,panel_size,random_mean_bal_acc,random_std_bal_acc,random_p95_bal_acc,random_max_bal_acc,random_mean_auc,random_p95_auc,random_max_auc
0,2,0.666223,0.140721,0.887121,0.964394,0.704292,0.939697,0.992424
1,3,0.729314,0.126613,0.921326,1.000000,0.776687,0.968258,1.000000
2,5,0.784777,0.114844,0.957159,0.990909,0.841915,0.992506,1.000000


In [54]:
selected_geo_summary = geo_cv_results_df[[
    "panel_name",
    "panel_size",
    "geo_mean_bal_acc",
    "geo_mean_auc",
]]

geo_selected_vs_random = selected_geo_summary.merge(
    geo_random_summary,
    on="panel_size",
    how="left",
)

geo_selected_vs_random

,panel_name,panel_size,geo_mean_bal_acc,geo_mean_auc,random_mean_bal_acc,random_std_bal_acc,random_p95_bal_acc,random_max_bal_acc,random_mean_auc,random_p95_auc,random_max_auc
0,onesplit_k2,2,0.955303,1.000000,0.666223,0.140721,0.887121,0.964394,0.704292,0.939697,0.992424
1,onesplit_k3,3,0.955303,1.000000,0.729314,0.126613,0.921326,1.000000,0.776687,0.968258,1.000000
2,onesplit_k5,5,0.964394,1.000000,0.784777,0.114844,0.957159,0.990909,0.841915,0.992506,1.000000
3,stable_k2,2,0.973485,1.000000,0.666223,0.140721,0.887121,0.964394,0.704292,0.939697,0.992424
4,stable_k3,3,0.973485,1.000000,0.729314,0.126613,0.921326,1.000000,0.776687,0.968258,1.000000
5,stable_k5,5,0.973485,0.998485,0.784777,0.114844,0.957159,0.990909,0.841915,0.992506,1.000000


In [55]:
geo_random_results_df.to_csv(
    "../data/processed/gse32863_random_panel_results.csv",
    index=False,
)

geo_selected_vs_random.to_csv(
    "../data/processed/gse32863_selected_vs_random_panels.csv",
    index=False,
)

In GSE32863, TCGA-discovered compact panels substantially outperformed the average random panel of the same size. The strongest evidence was observed at two genes: the stability-selected PYCR1–CD5L panel achieved 0.973 mean balanced accuracy and AUC 1.000, exceeding the 95th percentile and maximum of 200 random two-gene panels. As panel size increased, random panels more frequently approached selected-panel performance, indicating that high accuracy from moderate-sized bulk gene panels can arise partly from the abundance of tumor-normal signal in the transcriptome.

### Direct TCGA-to-GSE32863 transfer using contrast scores

In [56]:
directional_panels = {
    "stable_k2_contrast": {
        "tumor_up": ["PYCR1"],
        "normal_up": ["CD5L"],
    },
    "stable_k3_contrast": {
        "tumor_up": ["PYCR1"],
        "normal_up": ["CD5L", "ANGPT4"],
    },
    "stable_k5_contrast": {
        "tumor_up": ["PYCR1", "ETV4", "ADM2"],
        "normal_up": ["CD5L", "ANGPT4"],
    },
    "onesplit_k2_contrast": {
        "tumor_up": ["PYCR1", "ETV4"],
        "normal_up": [],
    },
    "onesplit_k3_contrast": {
        "tumor_up": ["PYCR1", "ETV4"],
        "normal_up": ["CD5L"],
    },
    "onesplit_k5_contrast": {
        "tumor_up": ["PYCR1", "ETV4", "ADM2", "LGR4"],
        "normal_up": ["CD5L"],
    },
}

In [57]:
def contrast_score(X_df, tumor_up, normal_up):
    tumor_score = X_df[tumor_up].mean(axis=1) if len(tumor_up) > 0 else 0
    normal_score = X_df[normal_up].mean(axis=1) if len(normal_up) > 0 else 0
    return tumor_score - normal_score

In [58]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

contrast_rows = []

for panel_name, parts in directional_panels.items():
    score = contrast_score(
        geo_gene_expr,
        tumor_up=parts["tumor_up"],
        normal_up=parts["normal_up"],
    )

    auc = roc_auc_score(geo_y, score)

    # choose threshold using median score as simple unsupervised cutoff
    threshold = np.median(score)
    y_pred = (score > threshold).astype(int)

    bal_acc = balanced_accuracy_score(geo_y, y_pred)

    contrast_rows.append({
        "panel_name": panel_name,
        "tumor_up": ",".join(parts["tumor_up"]),
        "normal_up": ",".join(parts["normal_up"]),
        "geo_auc": auc,
        "geo_bal_acc_median_threshold": bal_acc,
        "threshold": threshold,
        "confusion_matrix": confusion_matrix(geo_y, y_pred).tolist(),
    })

contrast_results_df = pd.DataFrame(contrast_rows)

contrast_results_df

,panel_name,tumor_up,normal_up,geo_auc,geo_bal_acc_median_threshold,threshold,confusion_matrix
0,stable_k2_contrast,PYCR1,CD5L,0.994352,0.965517,0.663000,"[[56, 2], [2, 56]]"
1,stable_k3_contrast,PYCR1,"CD5L,ANGPT4",0.997325,0.982759,0.657238,"[[57, 1], [1, 57]]"
2,stable_k5_contrast,"PYCR1,ETV4,ADM2","CD5L,ANGPT4",0.999108,0.982759,0.943626,"[[57, 1], [1, 57]]"
3,onesplit_k2_contrast,"PYCR1,ETV4",,0.998216,0.982759,7.303143,"[[57, 1], [1, 57]]"
4,onesplit_k3_contrast,"PYCR1,ETV4",CD5L,0.997919,0.982759,0.590030,"[[57, 1], [1, 57]]"
5,onesplit_k5_contrast,"PYCR1,ETV4,ADM2,LGR4",CD5L,0.999703,0.982759,0.795186,"[[57, 1], [1, 57]]"


This means a simple two-gene contrast discovered from TCGA separates GSE32863 tumor vs normal almost perfectly.

Repeated-split TCGA selection identified ultra-compact LUAD tumor-normal contrasts that transfer directly to an independent microarray cohort. The stable PYCR1–CD5L two-gene contrast achieved AUC 0.994 on GSE32863 without model retraining, while the three-gene PYCR1–CD5L–ANGPT4 contrast achieved AUC 0.997.